# Size does not matter: Sub-Billion VLM NanoChimera is All You Need!

This project addresses the modern challenge of Vision-Language Model (VLM) deployment by testing the scalability hypothesis: **Is massive parameter count necessary for effective visual reasoning?**

We construct and train **NanoChimera VLM**, a novel modular architecture designed to achieve high performance while strictly remaining **Sub-Billion** parameters ($\approx 999.99 \text{ Million}$ total deployed). Our goal is threefold: to provide a practical tutorial on building custom VLM architectures by training the critical Projector Layer, to rigorously evaluate its cognitive capabilities (grounding and hallucination), and to serve as a proof-of-concept for **real-time, edge-friendly multimodal AI**.


First of all, lets import the libraries and set a common seed 42 so the experiment is reproducible.

---
The outline of the notebook goes as following:

0) Data Loading and Augmentations
1) Model Architecture
2) Training Pipeline
3) Real Evaluation
4) Experiments
5) Conclusions & Results

In [1]:

# Required installations
# %pip install lmms-eval
# %pip install loguru

# Regular python related
import os
import json
import time
import pickle
import random
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from loguru import logger
from tqdm import tqdm
import json
from datetime import datetime


# Pytorch related
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split
# from lmms_eval.api.model import lmms


# HuggingFace related
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, AutoModel

def set_seed(seed, use_gpu = True, change_numpy_seed = False):
    random.seed(seed)
    # This may cause problems with CUDA
    if change_numpy_seed:
      np.random.seed(seed)
    torch.manual_seed(seed)
    if use_gpu:
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
USE_SEED = True
if USE_SEED:
    set_seed(SEED, torch.cuda.is_available())

## 0. Data Loading & Augmentations

Thanks to the already well curated efforts by meta with the LLaVa model, we do not need to preprocess the data at all, its just a plug and play clean dataset. Now it has to be said that if we wanted to artificially augment the size of the dataset this could easily be done throught common computer vision data augmentation techniques, however due to the sheer data volume we dispose. Nevertheless, for completeness, below we add a set of transformations that could be easily plugged into the Torch dataset API to perform data augmentation.

In [2]:
class VLM_Dataset(Dataset):
    def __init__(
        self,
        size=10000,
        filename="training_dataset.pkl",
        dataset="damerajee/Llava-pretrain-small",
        split="train",
        overwrite=False,
        seed=42,
        transform=None
    ):
        self.transform = transform

        if os.path.exists(filename) and not overwrite:
            with open(filename, "rb") as f:
                self.data = pickle.load(f)
        else:
            stream = load_dataset(
                dataset,
                split=split,
                streaming=True
            ).shuffle(buffer_size=size, seed=seed)

            self.data = []
            for i, row in enumerate(stream):
                if i >= size:
                    break
                self.data.append({
                    "image": row["image"],
                    "caption": row["answer"]
                })

            with open(filename, "wb") as f:
                pickle.dump(self.data, f)

    def __getitem__(self, idx):
        item = self.data[idx]

        image = item["image"]
        caption = item["caption"]

        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "caption": caption
        }

    def __len__(self):
        return len(self.data)


In [3]:
# Data augmentation transformations
transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224),
    torchvision.transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    torchvision.transforms.RandomHorizontalFlip(p=0.3),
    torchvision.transforms.RandomVerticalFlip(p=0.3),
    torchvision.transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2,hue=0.05),
    torchvision.transforms.RandomGrayscale(p=0.1),
    torchvision.transforms.RandomAdjustSharpness(1.5, p=0.2),
    torchvision.transforms.ToTensor(),
])

# Basic transformation
transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224),
    torchvision.transforms.ToTensor(),
])

In [4]:
# Example of applying transformations to K images
"""
dataset = VLM_Dataset(
    size=50,
    transform=transforms
)
loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)


# Get a batch
batch = next(iter(loader))
images = batch["image"]
captions = batch["caption"]
K = images.size(0)
cols = 4
rows = math.ceil(K / cols)

plt.figure(figsize=(cols * 4, rows * 4))

for i in range(K):
    img = images[i].permute(1, 2, 0)

    # If normalized, undo it (adjust if you used different stats)
    img = img * 0.5 + 0.5
    img = img.clamp(0, 1)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(img)
    plt.title(captions[i][:40], fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.show()

"""
None

## 1. Model Architecture

This is one of the single most important points of this project, the architecture of NanoChimera that will allow us to connect visual to textual tokens and thus allow the LLM to understand images. Therefore the visual adapter and its quality is the biggest factor determining the quality of this VLM.

The easiest implementation of this adapter is an MLP, and we can also relax the implicit assumption that all the image vision tokens should be used through the connector and passed down to the LLM, which is often not the case, some visual tokens have undoubtedly more importance than others.

In [5]:
class VisionConnector(nn.Module):
    """
    Maps vision encoder features -> LLM embedding space
    Supports a flexible number of hidden layers.
    """
    def __init__(self, vision_dim, llm_dim, hidden_dims=(4096,)):
        super().__init__()

        layers = []
        # Start with vision_dim
        current_dim = vision_dim

        # Add hidden layers dynamically
        for h_dim in hidden_dims:
            layers.append(nn.Linear(current_dim, h_dim))
            layers.append(nn.GELU())
            current_dim = h_dim

        # Add the final projection layer to llm_dim
        layers.append(nn.Linear(current_dim, llm_dim))

        self.proj = nn.Sequential(*layers)

    def forward(self, vision_feats):
        # vision_feats: (B, N, vision_dim)
        return self.proj(vision_feats)  # (B, N, llm_dim)


class NanoChimera(nn.Module):
    """
    LLaVA-style VLM:
    - Frozen vision encoder
    - Frozen LLM
    - Trainable connector
    """
    def __init__(self, vision_encoder, llm, connector):
        super().__init__()
        self.vision = vision_encoder
        self.llm = llm
        self.connector = connector

        # Freeze vision + LLM
        self.vision.requires_grad_(False)
        self.llm.requires_grad_(False)

    def forward(
        self,
        images,                 # preprocessed images
        input_ids,              # text token IDs (with <image>)
        image_token_id,         # int
        labels=None
    ):
        """
        Returns LLM outputs (loss + logits)
        """

        # ---- vision ----
        with torch.no_grad():
            vision_feats = self.vision(images).last_hidden_state
            # (B, N, vision_dim)

        # ---- connector ----
        visual_tokens = self.connector(vision_feats)
        # (B, N, llm_dim)

        # ---- text embeddings ----
        text_embeds = self.llm.get_input_embeddings()(input_ids)
        # (B, T, llm_dim)

        # ---- replace <image> token with visual tokens ----
        B = input_ids.size(0)
        full_embeds = []

        for b in range(B):
            idx = (input_ids[b] == image_token_id).nonzero()[0].item()
            merged = torch.cat([
                text_embeds[b, :idx],
                visual_tokens[b],
                text_embeds[b, idx+1:]
            ], dim=0)
            full_embeds.append(merged)

        full_embeds = torch.stack(full_embeds, dim=0)

        # ---- LLM ----
        outputs = self.llm(
            inputs_embeds=full_embeds,
            labels=labels
        )

        return outputs

## 2. Training Pipeline definition

Now we will define a simple data splitting (train/validation/test) strategy along the training loop, to add some interesting MLOps features that will improve the quality of the training experience, we will add logging, persistent information of runs and managing checkpoints to always save the best models comparing to past models.

So the section is split into the following subsections:

1) Data splitting
2) Model setup
3) Discussion on simple training evaluation metrics
4) Training loop definition
5) Training example (**Pretraining**)
6) Pipeline extension (**Supervised Fine-Tuning**)



---
Below we can find a simple example of a pipeline hyperparameters configuration

In [6]:
# This is a sample
EX_CONFIG = {
  # Data
  "dataset_size": 2048,
  "batch_size": 32,
  #"batch_size": 256,

  # Optimization algorithm
  "learning_rate": 9e-4,
  "weight_decay": 0.01,

  # Connector parameters
  "n_visual_tokens": 32,
  "connector_hidden_dims": (48828, ),

  # training
  "epochs": 10,
  "grad_accum_steps": 8,
  "warmup_ratio": 0.05,
  "max_grad_norm": 1.0,
  "scheduler_start_factor": 0.1,

  # logging
  "log_every": 32,
  "eval_every": 256,

}

### 2.1 Splitting Strategy and Data Preparation

For now we won't make use of automatic cross validation and rely on a simple train/test/validation split. Thanks to Pytorch data loaders we can shuffle when sampling to avoid overfitting weird patterns.


In [7]:
def split_dataset(dataset, train=0.8, val=0.1, test=0.1, seed=SEED):
    assert train + val + test == 1.0

    n = len(dataset)
    n_train = int(train * n)
    n_val = int(val * n)
    n_test = n - n_train - n_val

    generator = torch.Generator().manual_seed(seed)

    return random_split(
        dataset,
        [n_train, n_val, n_test],
        generator=generator
    )

# For Pytorch Dataloader API
def collate_fn(batch):
    # Keep PIL images for the processor
    img_transform = torchvision.transforms.Compose([
        torchvision.transforms.Resize((336, 336)),
        torchvision.transforms.Grayscale(num_output_channels=3),
        torchvision.transforms.ToTensor()
    ])
    return {"image": torch.stack([img_transform(item["image"]) for item in batch]), "caption": [item["caption"] for item in batch]}

# LOAD DATA
data = VLM_Dataset(size=EX_CONFIG["dataset_size"], overwrite=False)
# data = VLM_Dataset(size=EX_CONFIG["dataset_size"], overwrite=True)
train_dataset, val_dataset, test_dataset = split_dataset(data)

train_loader = DataLoader(
    train_dataset, batch_size=EX_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=EX_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=EX_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
)

### 2.2 Model setup

In [8]:
# UTILS FUNCTIONS
# IMAGE to VISUAL TOKEN
def encode_image(img, K=32):
    with torch.no_grad():
        inputs = vision_processor(images=img, return_tensors="pt").to(DEVICE)
        feats = vision_model.vision_model(**inputs).last_hidden_state  # shape: [1, seq_len, vision_dim]
        feats = feats[:, :K, :]  # truncate to K tokens
    return connector(feats)   # project to LLM dim

# BUILD MULTIMODAL EMBEDDINGS (TEXTUAL)
def build_inputs(img, text, K, device):
    B = img.size(0)
    vis_embeds = encode_image(img, K)  # [1, K, llm_dim]
    text_input = [f"{IMAGE_TOKEN} {c}" for c in text]
    ids = tokenizer(text_input, return_tensors="pt", padding=True, truncation=True).input_ids.to(DEVICE)
    text_embeds = llm.get_input_embeddings()(ids)
    
    image_token_id = tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)
    idx_list = (ids == image_token_id).nonzero(as_tuple=False)[:, 1]
    
    inputs_embeds_list = []
    attention_masks = []
    max_len = 0
    
    for b in range(B):
        idx = idx_list[b].item()

        embeds = torch.cat(
            [
                text_embeds[b, :idx],
                vis_embeds[b],
                text_embeds[b, idx + 1:],
            ],
            dim=0,
        )

        inputs_embeds_list.append(embeds)
        max_len = max(max_len, embeds.size(0))

    padded_inputs = []
    for embeds in inputs_embeds_list:
        seq_len = embeds.size(0)
        pad_len = max_len - seq_len

        if pad_len > 0:
            pad = torch.zeros(pad_len, embeds.size(-1), device=device)
            embeds = torch.cat([embeds, pad], dim=0)

        padded_inputs.append(embeds)

        attention_masks.append(
            torch.cat(
                [
                    torch.ones(seq_len, device=device),
                    torch.zeros(pad_len, device=device),
                ],
                dim=0,
            )
        )

    inputs_embeds = torch.stack(padded_inputs, dim=0)   # [B, L, D]
    attention_mask = torch.stack(attention_masks, dim=0)  # [B, L]

    return inputs_embeds, attention_mask, idx_list

In [9]:
# LOAD MODELS, LLM TOKENIZER, VISION PROCESSOR

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LLM_NAME = "Qwen/Qwen2.5-0.5B"
VISION_NAME = "google/siglip2-base-patch16-224"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME, dtype=torch.float32
).to(DEVICE)

vision_processor = AutoProcessor.from_pretrained(VISION_NAME, use_fast=True)
vision_model = AutoModel.from_pretrained(
    VISION_NAME, dtype=torch.float32
).to(DEVICE)

# ADD IMAGE TOKEN to LLM tokenizer vocabulary so it recognizes images
IMAGE_TOKEN = "<image>"
if IMAGE_TOKEN not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({"additional_special_tokens": [IMAGE_TOKEN]})
    llm.resize_token_embeddings(len(tokenizer))

# CONSTRUCT NANO CHIMERA MODEL (QWEN 2.5 + SIGILP + CONNECTOR)
# SigLIP Embedding output dimension
vision_dim = vision_model.config.vision_config.hidden_size
# LLM Embedding output dimension
llm_dim = llm.config.hidden_size

connector = VisionConnector(
    vision_dim=vision_dim,
    llm_dim=llm_dim,
    hidden_dims=EX_CONFIG["connector_hidden_dims"]
).to(DEVICE).float()
nano_chimera = NanoChimera(
    vision_encoder=vision_model.vision_model,
    llm=llm,
    connector=connector
).to(DEVICE)

NanoChimera

__main__.NanoChimera

### 2.3 Discussion on simple training evaluation metrics

As we already know LLM evaluation is a non-trivial task due to all the nuances of language, then multimodal LLM (MLLM) being a strict superset of these models, we can easily see the complexity grows as the number of modalities grows, strictly. Real performance evaluation is usually done through benchmarks and human/llm-as-a-judge evaluation, although for training such expensive evaluation metrics cannot be used.


So besides **negative log-likelihood (NLL) loss** or **cross-entropy loss** for raw token classification, we would benefit from an interpretable metric to evaluate (similar to accuracy, recall or f1), which for LLMs pretraining its usually either of these 2:

1) Token-Accuracy: measures the fraction of correctly predicted tokens, ignoring masked tokens (e.g. image tokens or padding), in other words: **Did the model’s argmax token match the label?**. Its easy to interpret but satures quickly and ignores near-misses and confidence!


$
\text{Token Accuracy}
=
\frac{1}{|\mathcal{M}|}
\sum_{t \in \mathcal{M}}
\mathbf{1}\!\left[ \hat{y}_t = y_t \right]
$

2) Perplexity: the exponential of the average negative log-likelihood. In other words: **On average, how many equally likely tokens the model is confused between.** When the PPL = 1, then the prediction is perfect, otherwise PPL = 5 its a random guess between 5 tokens, the lower the better.

$
\text{Perplexity}
=
\exp\!\left(
\mathcal{L}_{\mathrm{NLL}}
\right)
=
\exp\!\left(
-\frac{1}{|\mathcal{M}|}
\sum_{t \in \mathcal{M}}
\log p_\theta\!\left( y_t \mid x_{<t} \right)
\right)
$

So in conclusion, Token Accuracy can be used for sanity check and debugging, but to get an actual intuition of performance north-star we will use perplexity.

In [10]:
class LMStats:
    """Accumulates language modeling statistics."""
    def __init__(self):
        self.loss_sum = 0.0
        self.token_count = 0
        self.correct = 0

    def update(self, loss, logits, labels):
        """
        loss: scalar CE loss (already reduced)
        logits: [1, T, V]
        labels: [1, T] with -100 masked tokens
        """
        mask = labels != -100
        n_tokens = mask.sum().item()

        self.loss_sum += loss.item() * n_tokens
        self.token_count += n_tokens

        with torch.no_grad():
            preds = logits.argmax(dim=-1)
            self.correct += ((preds == labels) & mask).sum().item()

    def avg_loss(self):
        return self.loss_sum / max(self.token_count, 1)

    def perplexity(self):
        return math.exp(self.avg_loss())

    def accuracy(self):
        return self.correct / max(self.token_count, 1)


### 2.4 Training Pipeline definition

BIG TODO: The pipeline efficiceny can still be greatly improved, specially the auxilliary functions like build inputs and others which are not batched!

In [11]:
def train_one_epoch(
    model,
    loader,
    tokenizer,
    optimizer,
    scheduler,
    connector,
    criterion,
    device,
    EX_CONFIG,
    warmup_steps,
    global_step
):
    """
    One training epoch.
    Returns:
        avg_loss_per_token,
        perplexity,
        token_accuracy,
        total_tokens_processed_in_epoch,
        updated_global_step
    """
    model.train()
    stats = LMStats()

    optimizer.zero_grad()
    pbar = tqdm(loader, desc="Training", unit="batch")

    for batch in pbar:
        img = batch["image"].to(device)
        captions = batch["caption"]
        B = img.size(0)
        
        text = [f"{IMAGE_TOKEN} {c}" for c in captions]
        labels = tokenizer(text, return_tensors="pt", padding=True, truncation=True).input_ids.to(device)

        inputs_embeds, attention_mask, idx = build_inputs(
            img, text, EX_CONFIG["n_visual_tokens"], device
        )
        
        new_labels = []
        for b in range(B):
            lbl = torch.cat([
                labels[b, :idx[b]],
                torch.full((EX_CONFIG["n_visual_tokens"],), -100, device=device),
                labels[b, idx[b] + 1:]
            ], dim=0)
            
            new_labels.append(lbl)
        
        max_len = max(l.size(0) for l in new_labels)
        padded_labels = []
        for lbl in new_labels:
            pad_len = max_len - lbl.size(0)
            if pad_len > 0:
                lbl = torch.cat([lbl, torch.full((pad_len,), -100, device=device)], dim=0)
            padded_labels.append(lbl)
        
        labels = torch.stack(padded_labels, dim=0)  # shape: [B, max_len]
        
        max_len = labels.size(1)
        if inputs_embeds.size(1) < max_len:
            pad_len = max_len - inputs_embeds.size(1)
            pad = torch.zeros((B, pad_len, inputs_embeds.size(-1)), device=device)
            inputs_embeds = torch.cat([inputs_embeds, pad], dim=1)
        
            attention_mask = torch.cat(
                [attention_mask, torch.zeros((B, pad_len), device=device)],
                dim=1
            )
        elif inputs_embeds.size(1) > max_len:
            inputs_embeds = inputs_embeds[:, :max_len, :]

        outputs = model.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        
        raw_loss = criterion(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        )

        (raw_loss / EX_CONFIG["grad_accum_steps"]).backward()

        stats.update(raw_loss, logits, labels)

        global_step += 1

        if global_step % EX_CONFIG["grad_accum_steps"] == 0:
            torch.nn.utils.clip_grad_norm_(
                connector.parameters(),
                EX_CONFIG["max_grad_norm"]
            )
            optimizer.step()
            optimizer.zero_grad()

            if global_step < warmup_steps:
                scheduler.step()

        pbar.set_postfix(
            loss=f"{stats.avg_loss():.4f}",
            ppl=f"{stats.perplexity():.2f}",
            acc=f"{stats.accuracy():.4f}",
            tokens=f"{stats.token_count}"
        )

    return (
        stats.avg_loss(),
        stats.perplexity(),
        stats.accuracy(),
        stats.token_count,
        global_step
    )
@torch.no_grad()
def evaluate(
    model,
    loader,
    tokenizer,
    criterion,
    device,
    EX_CONFIG
):
    model.eval()
    stats = LMStats()

    pbar = tqdm(loader, desc="Evaluating", unit="batch")

    for batch in pbar:
        img = batch["image"].to(device)
        captions = batch["caption"]
        B = img.size(0)
        
        text = [f"{IMAGE_TOKEN} {c}" for c in captions]
        labels = tokenizer(text, return_tensors="pt", padding=True, truncation=True).input_ids.to(device)

        inputs_embeds, attention_mask, idx = build_inputs(
            img, text, EX_CONFIG["n_visual_tokens"], device
        )
        
        new_labels = []
        for b in range(B):
            lbl = torch.cat([
                labels[b, :idx[b]],
                torch.full((EX_CONFIG["n_visual_tokens"],), -100, device=device),
                labels[b, idx[b] + 1:]
            ], dim=0)
            
            new_labels.append(lbl)
        
        max_len = max(l.size(0) for l in new_labels)
        padded_labels = []
        for lbl in new_labels:
            pad_len = max_len - lbl.size(0)
            if pad_len > 0:
                lbl = torch.cat([lbl, torch.full((pad_len,), -100, device=device)], dim=0)
            padded_labels.append(lbl)
        
        labels = torch.stack(padded_labels, dim=0)  # shape: [B, max_len]
        
        max_len = labels.size(1)
        if inputs_embeds.size(1) < max_len:
            pad_len = max_len - inputs_embeds.size(1)
            pad = torch.zeros((B, pad_len, inputs_embeds.size(-1)), device=device)
            inputs_embeds = torch.cat([inputs_embeds, pad], dim=1)
        
            attention_mask = torch.cat(
                [attention_mask, torch.zeros((B, pad_len), device=device)],
                dim=1
            )
        elif inputs_embeds.size(1) > max_len:
            inputs_embeds = inputs_embeds[:, :max_len, :]

        outputs = model.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        
        raw_loss = criterion(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        )

        stats.update(raw_loss, logits, labels)

    return stats.avg_loss(), stats.perplexity(), stats.accuracy(), stats.token_count
def model_training(
    n_epochs,
    nano_chimera,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    model_name="best_nano_chimera.pt"
):
    """
    Standard training loop for multimodal LLMs.
    - Tracks loss, perplexity, token accuracy, token counts
    - Saves best checkpoint
    - Saves JSON + PNG + PDF report for each run
    """

    # 0. Experiment naming & folders
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    dataset_size = EX_CONFIG.get("dataset_size", "unk")
    batch_size = EX_CONFIG.get("batch_size", "unk")
    hidden = "x".join(map(str, EX_CONFIG["connector_hidden_dims"]))
    run_name = f"run_{timestamp}_ds{dataset_size}_bs{batch_size}_h{hidden}_ep{n_epochs}"

    output_dir = os.path.join("runs", run_name)
    os.makedirs(output_dir, exist_ok=True)

    best_model_path = os.path.join(output_dir, model_name)
    json_path = os.path.join(output_dir, "results.json")
    plot_path = os.path.join(output_dir, "training_metrics.pdf")
    png_path = os.path.join(output_dir, "training_metrics.png")
    pdf_path = os.path.join(output_dir, "report.pdf")

    # 1. Training bookkeeping
    best_val_loss = float("inf")

    train_losses, train_accs, train_ppls = [], [], []
    val_losses, val_accs, val_ppls = [], [], []
    train_token_counts, val_token_counts = [], []

    global_step = 0

    # 2. Training loop
    for epoch in range(n_epochs):
        start = time.time()

        train_loss, train_ppl, train_acc, train_tokens, global_step = train_one_epoch(
            nano_chimera,
            train_loader,
            tokenizer,
            optimizer,
            scheduler,
            connector,
            criterion,
            device,
            EX_CONFIG,
            warmup_steps,
            global_step
        )

        val_loss, val_ppl, val_acc, val_tokens = evaluate(
            nano_chimera,
            val_loader,
            tokenizer,
            criterion,
            device,
            EX_CONFIG
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(nano_chimera.state_dict(), best_model_path)

        logger.info(
            f"Epoch {epoch+1}/{n_epochs} | "
            f"Train Loss {train_loss:.4f} | PPL {train_ppl:.2f} | Acc {train_acc:.4f} | Tokens {train_tokens} | "
            f"Val Loss {val_loss:.4f} | PPL {val_ppl:.2f} | Acc {val_acc:.4f} | Tokens {val_tokens} | "
            f"Time {time.time()-start:.1f}s"
        )

        train_losses.append(train_loss)
        train_accs.append(train_acc)
        train_ppls.append(train_ppl)
        train_token_counts.append(train_tokens)

        val_losses.append(val_loss)
        val_accs.append(val_acc)
        val_ppls.append(val_ppl)
        val_token_counts.append(val_tokens)

    # 3. Plot metrics (Token count, accuracy, perplexity, loss...)
    train_token_cum = np.cumsum(train_token_counts)
    val_token_cum = np.cumsum(val_token_counts)

    fig = plt.figure(figsize=(20,5))

    plt.subplot(1,4,1)
    plt.plot(train_losses, label="Train")
    plt.plot(val_losses, label="Val")
    plt.title("Loss")
    plt.grid(True)
    plt.legend()

    plt.subplot(1,4,2)
    plt.plot(train_accs, label="Train")
    plt.plot(val_accs, label="Val")
    plt.title("Token Accuracy")
    plt.grid(True)
    plt.legend()

    plt.subplot(1,4,3)
    plt.plot(train_token_cum, label="Train")
    plt.plot(val_token_cum, label="Val")
    plt.title("Cumulative Tokens")
    plt.grid(True)
    plt.legend()

    plt.subplot(1,4,4)
    plt.plot(train_ppls, label="Train")
    plt.plot(val_ppls, label="Val")
    plt.title("Perplexity")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.savefig(plot_path, dpi=150)
    plt.savefig(png_path, dpi=150)

    plt.close()

    # 4. Save RUN Report JSON
    run_summary = {
        "run_name": run_name,
        "timestamp": timestamp,
        "config": EX_CONFIG,
        "metrics": {
            "train_loss": train_losses,
            "val_loss": val_losses,
            "train_accuracy": train_accs,
            "val_accuracy": val_accs,
            "train_perplexity": train_ppls,
            "val_perplexity": val_ppls,
            "train_tokens_epoch": train_token_counts,
            "val_tokens_epoch": val_token_counts,
            "train_tokens_cumulative": train_token_cum.tolist(),
            "val_tokens_cumulative": val_token_cum.tolist(),
        },
        "artifacts": {
            "plot": plot_path,
            "best_model": best_model_path,
            "pdf_report": pdf_path
        },
        "best_val_loss": best_val_loss
    }

    with open(json_path, "w") as f:
        json.dump(run_summary, f, indent=2)

    # 5. Generate PDF report
    with PdfPages(pdf_path) as pdf:
        fig = plt.figure(figsize=(11,8))


        plt.text(0.01, 0.95, "Final Metrics Summary", fontsize=14)

        summary_lines = [
            f"Final Train Loss: {train_losses[-1]:.4f}",
            f"Final Val Loss:   {val_losses[-1]:.4f}",
            f"Final Train Acc:  {train_accs[-1]:.4f}",
            f"Final Val Acc:    {val_accs[-1]:.4f}",
            f"Final Train PPL:  {train_ppls[-1]:.2f}",
            f"Final Val PPL:    {val_ppls[-1]:.2f}",
            "",
            f"Total Train Tokens: {int(train_token_cum[-1])}",
            f"Total Val Tokens:   {int(val_token_cum[-1])}",
            "",
            f"Best Validation Loss: {best_val_loss:.4f}"
        ]

        y = 0.88
        for line in summary_lines:
            plt.text(0.01, y, line, fontsize=11)
            y -= 0.05

        y = 0.85
        for k, v in EX_CONFIG.items():
            plt.text(0.01, y, f"{k}: {v}", fontsize=9)
            y -= 0.03

        plt.axis("off")
        pdf.savefig(fig)
        plt.close()

        img = plt.imread(png_path)
        fig = plt.figure(figsize=(11,5))
        plt.imshow(img)
        plt.axis("off")
        pdf.savefig(fig)
        plt.close()

    logger.info(f"Run saved to {output_dir}")

    return (
        train_losses,
        train_accs,
        val_losses,
        val_accs,
        train_token_counts,
        val_token_counts
    )


### 2.5 Example (Train + Inference)


In [ ]:

criterion = nn.CrossEntropyLoss(ignore_index=-100,
    label_smoothing=0.05)
criterion = criterion.to(DEVICE)

optimizer = torch.optim.AdamW(
    connector.parameters(),
    lr=EX_CONFIG["learning_rate"],
    weight_decay=EX_CONFIG["weight_decay"]
)

total_steps = len(data) // EX_CONFIG["grad_accum_steps"]
warmup_steps = int(EX_CONFIG["warmup_ratio"] * total_steps)

scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=EX_CONFIG["scheduler_start_factor"],
    total_iters=warmup_steps
)

connector.train()
optimizer.zero_grad()

train_losses, train_accs, val_losses, val_accs, train_token_counts, val_token_counts = model_training(
    n_epochs=EX_CONFIG["epochs"],
    nano_chimera=nano_chimera,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=DEVICE,
    model_name="best_nano_chimera.pt"
)

Training:  14%|████▏                        | 1/7 [00:07<00:43,  7.23s/batch, acc=0.0072, loss=12.7799, ppl=355008.06, tokens=832]

#### Inference!

This is a big TODO, i'm not sure this is working properly

In [29]:
@torch.no_grad()
def inference_nano_chimera(
    nano_chimera,
    dataloader,
    tokenizer,
    connector,
    K,
    device,
    max_new_tokens=20
):
    """
    Standard inference function for a NanoChimera.

    Args:
        nano_chimera: the NanoChimera model
        dataloader: DataLoader with batch of {"image": [...], "caption": [...]}
        tokenizer: HuggingFace tokenizer
        connector: VisionConnector
        K: number of visual tokens
        device: 'cuda' or 'cpu'
        max_new_tokens: generation length

    Returns:
        List of dicts: [{"image": img, "caption": gt_caption, "generated": pred_caption}, ...]
    """
    nano_chimera.eval()
    results = []

    for batch in dataloader:
        images = batch["image"]
        captions = batch["caption"]
        batch_size = len(images)

        for i in range(batch_size):
            img = images[i]
            gt_caption = captions[i]

            # ---- encode image ----
            inputs = vision_processor(images=img, return_tensors="pt").to(device)
            vis_feats = vision_model.vision_model(**inputs).last_hidden_state[:, :K, :]
            vis_embeds = connector(vis_feats)  # [1, K, llm_dim]

            # ---- build input embeddings ----
            text_input = f"{IMAGE_TOKEN} What is in this image?"  # prompt can be added here
            ids = tokenizer(text_input, return_tensors="pt").input_ids.to(device)
            text_embeds = nano_chimera.llm.get_input_embeddings()(ids)

            # Find IMAGE_TOKEN index
            idxs = (ids == tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)).nonzero(as_tuple=False)
            if len(idxs) != 1:
                raise ValueError("Expected exactly one IMAGE_TOKEN in input")
            idx = idxs[0, 1]

            # Concatenate embeddings
            input_embeds = torch.cat(
                [text_embeds[:, :idx], vis_embeds, text_embeds[:, idx+1:]],
                dim=1
            )
            attention_mask = torch.ones(input_embeds.size()[:-1], device=device)

            # ---- generate ----
            generated_ids = nano_chimera.llm.generate(
                inputs_embeds=input_embeds,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                min_new_tokens=10,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.8,
                top_p=0.9,
                repetition_penalty=1.15,
                no_repeat_ngram_size=2,
            )

            prompt_len = input_embeds.size(1)
            gen_tokens = generated_ids[0, prompt_len:]
            pred_caption = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

            results.append({
                "image": img,
                "caption": gt_caption,
                "generated": pred_caption
            })

    return results

In [30]:
test_loader = DataLoader(
    test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn
)

results = inference_nano_chimera(
    nano_chimera=nano_chimera,
    dataloader=test_loader,
    tokenizer=tokenizer,
    connector=connector,
    K=EX_CONFIG["n_visual_tokens"],
    device=DEVICE,
    max_new_tokens=20,
)

# Print first 5 results
for r in results[:5]:
    print("GT:", r["caption"])
    print("GEN:", r["generated"])
    # plt.imshow(r["image"])
    # plt.axis('off')
    # plt.show()

GT: the brain cerebral and cerebral impairment
GEN:  of  the and good one well with bad condition
GT: the under ground view, from the street
GEN:  from of  is in an

The air air space
GT: the roundabout near to the station in hackney where the fire started
GEN:   is. of a in for from the (
GT: the white toyota capri at the cheshire motor museum
GEN:  (1 2( )3,4...
GT: a cpu processor sits on top of the computer chips
GEN:  is the of  a s s k

s


### 2.6 Pipeline Extension

In typical VLM training scenarios, analogously to LLM training, there are multiple phases not only a single iteration loop. For LLMs this is typically decomposed in the stages:

1. Pretraining: Similar to our training, the model learns to predict next tokens from **unsupervised corpora**, learning representations of language along the way.

2. Supervised Fine-Tuning: Through a **golden standard supervised Q&A dataset** the model learns to be a helpful assistant and to follow instructions.

3. Preference alingment (Reinforcement Learning from Human Feedback): Further refinement with Q&A style data with RL based algorithms like PPO, DPO, GRPO...

> However in multimodal VLM's the pipeline usually goes as following:

1. **Stage 1: Vision-Language Feature Alignment (Pre-training)**: In this phase the objective is align visual features from Visual Encoders with Language Model embeddings. Typical configurations of hyperparameters are:

    1-2 epochs on ~600K image-text pairs Batch Size: 256-512, Learning Rate: 1e-3 to 2e-3 (Bigger)

2. **Stage 2: Visual Instruction Tuning (SFT):**
  In this phase the objective is to teach the model to follow multimodal instructions. Typical configurations of hyperparameters are:

    3-5 epochs on ~150K instruction-examples Batch Size: 128-256 Learning Rate: 2e-5 (much smaller).



---
So if we want to refine the quality of our model to go beyond a simple pretraining modality alignment, we need to perform yet another training using visual instruction tuning datasets. Luckily for us again, huggingface contains such already processed datasets. For example: liuhaotian/LLaVA-Instruct-150K.


In [ ]:
# This is a sample
VSFT_CONFIG = {
  # Data
  "dataset_size": 16384,
  "batch_size": 128,

  # Optimization algorithm
  "learning_rate": 2e-5,
  "weight_decay": 0.01,

  # Connector parameters
  "n_visual_tokens": 32,
  "connector_hidden_dims": (4096, ),

  # training
  "epochs": 3,
  "grad_accum_steps": 8,
  "warmup_ratio": 0.05,
  "max_grad_norm": 1.0,
  "scheduler_start_factor": 0.1,

  # logging
  "log_every": 128,
  "eval_every": 256,

}

In [ ]:
# LOAD DATA
# TODO: This data format is weird, maybe it should be adapted in the collate_fn
data = VLM_Dataset(dataset="liuhaotian/LLaVA-Instruct-150K", split="train", size=VSFT_CONFIG["dataset_size"], overwrite=True)
train_dataset, val_dataset, test_dataset = split_dataset(data)

train_loader = DataLoader(
    train_dataset, batch_size=VSFT_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=VSFT_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=VSFT_CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn
)

In [ ]:
# TRAINING PIPELINE

## 3. Evaluation pipeline

1. pip install lmms-eval
2. Write tiny LMMSModel wrapper
3. Call evaluator.simple_evaluate(...)
4. Start with vqa_v2_val + limit=100

In [ ]:
#!lmms-eval --help

## 4. Experiments

In this section we are going to execute some experiments to try to optimize this NanoChimera architecture as much as possible with different configurations of hyperparameters and even small architecture size changes. Given this multi-variable optimization landscape, we need to take into account all of the hyperparameters and factors that can affect the model performance.

### 4.1 Basic Adapter (1 layer, 4096 hidden units)

In [ ]:
config = {
    "DATASET_SIZE": 10000,
    "BATCH_SIZE": 16,
    "NUM_EPOCHS": 20,
    "LEARNING_RATE": 1e-5
}

## 4.2 2-Layer Adapter



In [ ]:
config = {
    "DATASET_SIZE": 10000,
    "BATCH_SIZE": 16,
    "NUM_EPOCHS": 20,
    "LEARNING_RATE": 1e-5
}

## 5. Results

In this section we are going to discuss the different experiment results, as well as the winning model pipeline combination.

Also we are going to upload it to hugging-face to opensource it for everybody to use freely, although it lacks some refinement.


In [ ]:
# Some token to upload to huggingface, adrian i trust in you you won't upload racist memes to my huggignface account :)


model = None
tokenizer = None

repo_name = "NanoChimeraVLM-V1"
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)


 ## 5. Conclusions